# 证据链 B：岗位验收演示
使用团队真实模型记录离线重放；CPU 即可，不需要密钥。

In [ ]:
import base64,io,tempfile,zipfile,subprocess,sys,json
from pathlib import Path
root=Path(tempfile.mkdtemp())
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z: z.extractall(root)
b=root/'modules/evidence_b_checker'

## 第一步：用团队的提案、原句和工单跑一次
图、答复和成绩单应当对应同一次运行。

In [ ]:
def run(script):
    r=subprocess.run([sys.executable,str(b/script)],cwd=root,text=True,capture_output=True)
    print(r.stdout,r.stderr)
    r.check_returncode()
run('acceptance_pr5.py')
bundle=json.loads((b/'examples/pr5/replayed_bundle.json').read_text(encoding='utf8'))
print('故障开始：',bundle['task']['event_time'])
print('提交时间：',next(r for r in bundle['tool_receipts'] if r['source_type']=='TICKET_DB')['records'][0]['submitted_at'])
print('成绩单：',bundle['scorecard']['level'])

## 第二步：检查迟交、到点、无权限和错误工单等情况
这里检查合成结果是否符合预期；不需要训练模型。

In [ ]:
run('verify_pr5_checker.py')